# Asset Brand ETL & Data Cleaning
### AMS Legacy Migration to Smart Microsoft SQL Server (`brands` table)

---

## 1. Overview & Business Context
This notebook performs data cleaning, normalization, relational mapping, deduplication, and product relegation on the legacy AMS (Asset Management System) brands dataset (`data/ams/tb_assets_brand_202609301418.csv`).

The goal is to prepare clean, verified data ready for direct import into the Microsoft SQL Server **`brands`** table for the new **Smart** application.

### Key Business Transformations:
1. **Removal of `create_owner`**: Dropped to optimize storage as SMART does not track brand creator users.
2. **Corporate Brand Casing & Acronym Preservation**: Standardize brand names into proper Title Case while strictly respecting corporate acronyms and branding guidelines (e.g., `HP`, `LG`, `IBM`, `MSI`, `ASUS`, `3Com`, `D-Link`, `TP-Link`, `DJI`, `APC`, `QNAP`, `TOA`, `V-Gen`, `GoPro`, `ViewSonic`, `SanDisk`, `TaffGUARD`, `OneMed`, `BenQ`, `ETAP`, `AVEVA`, `Fortinet`, `DIY`).
3. **Sentence Case Description Formatting**: Properly format all descriptions to clean sentence case while preserving standard acronyms (e.g., `DVR & CCTV`, `AC split`, `AC portable`, `AC AHU`, `Storage, HDD, SSD`, `UPS, rak server`, `Kotak P3K`, `Monitor LED`).
4. **Typo Resolution & Duplication Merges**:
   - `FULTURA` (id: 3118) $\rightarrow$ `FUTURA` (id: 2088)
   - `SAKEN` (id: 3136) $\rightarrow$ `SANKEN` (id: 3104)
   - `TOA CORPORATION` (id: 3111) $\rightarrow$ `TOA` (id: 59)
   - Duplicate dummy entry `3` (id: 3156) $\rightarrow$ primary `3` (id: 3)
   - `CUSTOM` (id: 3099) $\rightarrow$ merged into `Rakitan` (id: 6) as an actual brand for assembled PCs
   - `ONLINE` (id: 3107) $\rightarrow$ merged into `No Brand` (id: 3129) as the actual brand for unbranded assets
   - `LCEX S03` (id: 3159) $\rightarrow$ consolidated into `Pepperl+Fuchs` (`SMART EX03`, id: 3157)
   - Typo spelling corrections (`INFOKUS` $\rightarrow$ `InFocus`, `BUFALLO` $\rightarrow$ `Buffalo`, `CUMINS` $\rightarrow$ `Cummins`, `Ferrarti Ferro` $\rightarrow$ `Ferratti Ferro`).
5. **Product Model Relegation**:
   - `ACROBAT PRO` (id: 67) relegated under `Adobe` (id: 38)
   - `PAJERO` (id: 3174) relegated under `Mitsubishi` (id: 3161)
   - `PANABOARD` (id: 23) relegated under `Panasonic` (id: 28)
   - `CAESAR` (id: 37) relegated under `Intergraph` (id: 64)
   - `FINGERPRINT M100` (id: 18) relegated under `Solution` (id: 3140)
6. **Special Classification Alignments**:
   - `China` (id: 56), `Rakitan` (id: 6), `DIY` (id: 3128), `No Brand` (id: 3129), `EPY Industri` (id: 3101), `JAP Energia` (id: 3103), `Fortinet` (id: 3110, renamed from Fortigate), and `Pepperl+Fuchs` (id: 3157, absorbs 3159) are treated as **actual brands** (`not_actual_brand = False`).
7. **Staging Audit Columns**:
   - `ams_id`: Stores the legacy AMS ID(s) associated with each brand (including merged/relegated IDs as an array list).
   - `not_actual_brand`: Boolean flag (`True`/`False`). Correct, verified brands have `False`. Relegated product models, categories (`Building`, `DRY CONTAINER`), or dummy test records (`3`, `TEST`) are flagged as `True` for review with the asset team.
8. **Strict Clean Master Export**:
   - Only confirmed, actual brands (`not_actual_brand == False`) are exported to final production outputs (`final/brand/brands.csv` and `final/brand/brands.sql`).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------
# Dynamic Path Configurations
# Robustly resolves data directory regardless of whether
# current working directory is /home/jovyan, /home/jovyan/work,
# or the repository root.
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path("data"),                                  # Running from project root
        Path("/home/jovyan/data"),                      # Running inside docker container
        Path("../data"),                               # Running from notebooks/ directory
        Path("/home/ran/Devs/ams-smart-migration/data") # Absolute host path
    ]
    for p in candidates:
        if p.exists() and (p / "ams").exists():
            return p.resolve()
    return Path("data").resolve()

DATA_DIR = get_data_dir()
RAW_DATA_PATH = DATA_DIR / "ams" / "tb_assets_brand_202609301418.csv"
STAGING_DIR = DATA_DIR / "smart" / "staging" / "brand"
FINAL_DIR = DATA_DIR / "smart" / "final" / "brand"

STAGING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory:    {DATA_DIR}")
print(f"Raw AMS Brand CSV: {RAW_DATA_PATH} (exists: {RAW_DATA_PATH.exists()})")
print(f"Staging directory: {STAGING_DIR}")
print(f"Final directory:   {FINAL_DIR}")


Data directory:    /home/jovyan/data
Raw AMS Brand CSV: /home/jovyan/data/ams/tb_assets_brand_202609301418.csv (exists: True)
Staging directory: /home/jovyan/data/smart/staging/brand
Final directory:   /home/jovyan/data/smart/final/brand


In [2]:
# Load raw brand dataset, treating literal string 'NULL', 'null', and blanks as NaN
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    na_values=["NULL", "null", "NaN", "nan", ""],
    keep_default_na=True,
    dtype=str
)

# Strip whitespace from string columns
for col in df_raw.select_dtypes(include=["object"]).columns:
    df_raw[col] = df_raw[col].str.strip()

print(f"Loaded {len(df_raw)} total rows from AMS brand dataset.")
display(df_raw.head(10))


Loaded 188 total rows from AMS brand dataset.


,id,brand,description,create_date,create_owner
0,1,COMPAQ,NaN,2016-12-21 09:17:47.923,302716
1,2,INFOKUS,NaN,2016-12-21 09:17:47.923,30716
2,3,3,208.100.0.117,2025-02-15 01:52:21.000,215050
3,4,YAMAHA,NaN,2016-12-21 09:17:47.923,30716
4,5,DLINK,NaN,2016-12-21 09:17:47.923,30716
5,6,Rakitan,NaN,2016-12-21 09:17:47.923,30716
6,7,DELL,NaN,2016-12-21 09:17:47.923,30716
7,8,TOSHIBA,NaN,2016-12-21 09:17:47.923,30716
8,9,FUJITSU,NaN,2016-12-21 09:17:47.923,30716
9,10,HP,NaN,2016-12-21 09:17:47.923,30716


## 3. Exploratory Analysis & Anomaly Detection

We examine:
1. Exact and case-insensitive duplicates.
2. Missing values across all columns.
3. Anomaly patterns such as IP addresses or dummy test records in `brand` or `description`.


In [3]:
# 1. Check for duplicates in brand names (case-insensitive)
brand_counts = df_raw["brand"].str.upper().value_counts()
duplicate_brands = brand_counts[brand_counts > 1]
print("=== Duplicate Brand Names in Raw AMS (Case-Insensitive) ===")
print(duplicate_brands)

# 2. Check null counts
print("\n=== Missing Values Summary ===")
print(df_raw.isna().sum())

# 3. Detect anomaly records (numeric dummy brands, test records, IP descriptions)
anomalies = df_raw[
    df_raw["brand"].str.contains(r"^\d+$|^TEST", case=False, na=False) |
    df_raw["description"].str.contains(r"\d+\.\d+\.\d+\.\d+", na=False)
]
print(f"\nDetected {len(anomalies)} obvious anomaly/test records:")
display(anomalies)


=== Duplicate Brand Names in Raw AMS (Case-Insensitive) ===
brand
3    2
Name: count, dtype: int64

=== Missing Values Summary ===
id                0
brand             0
description     118
create_date       0
create_owner      0
dtype: int64

Detected 4 obvious anomaly/test records:


,id,brand,description,create_date,create_owner
2,3,3,208.100.0.117,2025-02-15 01:52:21.000,215050
75,78,TEST,TESTA,2018-03-21 11:03:52.000,302716
154,3149,SHINPO,208.100.0.117,2025-02-15 01:52:47.000,215050
161,3156,3,208.100.0.117,2025-02-14 16:50:31.000,215050


## 4. Master Brand Configuration, Cleaning Rules, & Casing Strategy

### Business Transformation Rules:
1. **Corporate Casing & Special Formatting**: Acronyms and established brand casing patterns (`HP`, `LG`, `IBM`, `MSI`, `ASUS`, `3Com`, `D-Link`, `TP-Link`, `DJI`, `APC`, `QNAP`, `TOA`, `V-Gen`, `GoPro`, `ViewSonic`, `SanDisk`, `TaffGUARD`, `OneMed`, `BenQ`, `ETAP`, `AVEVA`, `Fortinet`, `DIY`).
2. **Sentence Case Description Formatting**: Properly format all descriptions to clean sentence case while preserving standard acronyms (e.g., `DVR & CCTV`, `AC split`, `AC portable`, `AC AHU`, `Storage, HDD, SSD`, `UPS, rak server`, `Kotak P3K`, `Monitor LED`, `Meja kerja`, `Kursi jaring`).
3. **Duplicate & Typo Merges**:
   - `FULTURA` (id: 3118) $\rightarrow$ `FUTURA` (id: 2088).
   - `SAKEN` (id: 3136) $\rightarrow$ `SANKEN` (id: 3104).
   - `TOA CORPORATION` (id: 3111) $\rightarrow$ `TOA` (id: 59).
   - `3` (id: 3156) $\rightarrow$ `3` (id: 3).
   - `CUSTOM` (id: 3099) $\rightarrow$ merged into `Rakitan` (id: 6).
   - `ONLINE` (id: 3107) $\rightarrow$ merged into `No Brand` (id: 3129).
   - Typo corrections for standalone brands: `INFOKUS` $\rightarrow$ `InFocus`, `BUFALLO` $\rightarrow$ `Buffalo`, `CUMINS` $\rightarrow$ `Cummins`, `Ferrarti Ferro` $\rightarrow$ `Ferratti Ferro`.
4. **Product Relegations**:
   - `ACROBAT PRO` (id: 67) relegated under `Adobe` (id: 38).
   - `PAJERO` (id: 3174) relegated under `Mitsubishi` (id: 3161).
   - `PANABOARD` (id: 23) relegated under `Panasonic` (id: 28).
   - `CAESAR` (id: 37) relegated under `Intergraph` (id: 64).
   - `FINGERPRINT M100` (id: 18) relegated under `Solution` (id: 3140).
5. **Actual Brands vs Non-Actual Brands Alignment**:
   - **Treated as Actual Brands** (`not_actual_brand = False`): `China` (id: 56), `Rakitan` (id: 6, absorbs 3099), `No Brand` (id: 3129, absorbs 3107), `DIY` (id: 3128), `EPY Industri` (id: 3101), `JAP Energia` (id: 3103), `Fortinet` (id: 3110, renamed from Fortigate), `Pepperl+Fuchs` (id: 3157, absorbs 3159), `Solution` (id: 3140, absorbs 18), `Adobe` (id: 38, absorbs 67), `Mitsubishi` (id: 3161, absorbs 3174), `Panasonic` (id: 28, absorbs 23), `Intergraph` (id: 64, absorbs 37), etc.
   - **Non-Actual Brands Flagged in Staging** (`not_actual_brand = True`):
     - Categories: `Building` (id: 3088), `DRY CONTAINER` (id: 3098).
     - Dummy / test entries: `3` (id: 3), `TEST` (id: 78).
     - Relegated products: `Acrobat Pro`, `Pajero`, `Panaboard`, `Caesar`, `Fingerprint M100`.


In [4]:
# ---------------------------------------------------------
# 1. Special Name Casing & Official Corporate Branding
# ---------------------------------------------------------
SPECIAL_NAME_CASING = {
    # Acronyms & Short Initials
    "HP": "HP",
    "LG": "LG",
    "IBM": "IBM",
    "MSI": "MSI",
    "ASUS": "ASUS",
    "NEC": "NEC",
    "DJI": "DJI",
    "APC": "APC",
    "QNAP": "QNAP",
    "TOA": "TOA",
    "TOA CORPORATION": "TOA",
    "GTC": "GTC",
    "AFES": "AFES",
    "SUNIX": "SUNIX",
    "ETAP": "ETAP",
    "AVEVA": "AVEVA",
    "AKASO": "AKASO",
    "BJI": "BJI",
    "GM": "GM",
    "JS": "JS",
    "ASV": "ASV",
    "DIY": "DIY",
    "AUX": "AUX",
    "Aux": "AUX",
    "JBL": "JBL",

    # Hyphenated, CamelCase & Multi-word Brands
    "3COM": "3Com",
    "DLINK": "D-Link",
    "TPLINK": "TP-Link",
    "VGEN": "V-Gen",
    "Go Pro": "GoPro",
    "Viewsonic": "ViewSonic",
    "Sandisk": "SanDisk",
    "TAFFGUARD": "TaffGUARD",
    "Onemed": "OneMed",
    "Benq": "BenQ",
    "BENQ": "BenQ",
    "MIKROTIK": "MikroTik",
    "NORTHBAYOU": "North Bayou",
    "Ferrarti Ferro": "Ferratti Ferro",
    "MCQUAY": "McQuay",
    "Lion Star": "Lion Star",
    "Western Digital": "Western Digital",
    "Altec Lansing": "Altec Lansing",
    "ALTEC LANSING": "Altec Lansing",
    "Hong Chang": "Hong Chang",
    "HONG CHANG": "Hong Chang",
    "HONG CHANG ": "Hong Chang",
    "Daya Kurnia": "Daya Kurnia",
    "Syly Art": "Syly Art",
    "Pongs Home": "Pongs Home",
    "PONGS HOME": "Pongs Home",
    "Fortigate": "Fortinet",
    "FORTIGATE": "Fortinet",
    "Fortinet": "Fortinet",
    "CHINA": "China",

    # Standalone Typo Corrections
    "INFOKUS": "InFocus",
    "BUFALLO": "Buffalo",
    "CUMINS": "Cummins",

    # Relegated & Non-Brand Formatting
    "ACROBAT PRO": "Acrobat Pro",
    "PAJERO": "Pajero",
    "PANABOARD": "Panaboard",
    "CAESAR": "Caesar",
    "FINGERPRINT M100": "Fingerprint M100",
    "SMART EX03": "Pepperl+Fuchs",
    "LCEX S03": "Pepperl+Fuchs",
    "Pepperl+Fuchs": "Pepperl+Fuchs",
    "DRY CONTAINER": "Dry Container",
    "EPY Industri": "EPY Industri",
    "JAP ENERGIA": "JAP Energia",
    "NOBRAND": "No Brand",
    "ONLINE": "No Brand"
}

def clean_brand_name(raw_name):
    if pd.isna(raw_name):
        return ""
    s = str(raw_name).strip()
    if s in SPECIAL_NAME_CASING:
        return SPECIAL_NAME_CASING[s]
    for k, v in SPECIAL_NAME_CASING.items():
        if s.upper() == k.upper():
            return v
    return s.title()

# ---------------------------------------------------------
# 2. Duplicate Merges (source_ams_id -> target_ams_id)
# ---------------------------------------------------------
MERGE_TARGET_MAP = {
    3118: 2088,   # FULTURA -> FUTURA
    3136: 3104,   # SAKEN -> SANKEN
    3111: 59,     # TOA CORPORATION -> TOA
    3156: 3,      # '3' (2nd instance) -> '3' (1st instance)
    3099: 6,      # CUSTOM -> Rakitan
    3107: 3129,   # ONLINE -> No Brand
    3159: 3157    # LCEX S03 -> Pepperl+Fuchs (SMART EX03)
}

# ---------------------------------------------------------
# 3. Product Relegation (product_ams_id -> parent_brand_ams_id)
# ---------------------------------------------------------
RELEGATE_PRODUCT_MAP = {
    67: 38,       # ACROBAT PRO -> ADOBE
    3174: 3161,   # PAJERO -> MITSUBISHI
    23: 28,       # PANABOARD -> PANASONIC
    37: 64,       # CAESAR -> INTERGRAPH
    18: 3140      # FINGERPRINT M100 -> SOLUTION
}

# ---------------------------------------------------------
# 4. Non-Actual Brands (Flagged as not_actual_brand = True)
# ---------------------------------------------------------
NON_ACTUAL_BRAND_REASONS = {
    3: "Dummy data / IP address (208.100.0.117)",
    3156: "Dummy data / IP address (208.100.0.117)",
    78: "Dummy test record (TEST / TESTA)",
    3088: "Asset category (Building / Gedung)",
    3098: "Asset category (DRY CONTAINER / Tempat Kerja RTU)",
    18: "Biometric hardware model (FINGERPRINT M100 relegated to Solution)",
    67: "Software product (ACROBAT PRO relegated to Adobe)",
    3174: "Vehicle model (PAJERO relegated to Mitsubishi)",
    23: "Electronic whiteboard model (PANABOARD relegated to Panasonic)",
    37: "Engineering software (CAESAR relegated to Intergraph)"
}

# ---------------------------------------------------------
# 5. Sentence Case Description Formatting & Overrides
# ---------------------------------------------------------
DESCRIPTION_SENTENCE_CASE = {
    # Acronyms & technical equipment phrases
    "DVR & CCTV": "DVR & CCTV",
    "Meja Kerja": "Meja kerja",
    "Kursi Jaring": "Kursi jaring",
    "KURSI HITAM": "Kursi hitam",
    "TEMPAT SAMPAH": "Tempat sampah",
    "JAM DINDING": "Jam dinding",
    "MEJA LIPAT": "Meja lipat",
    "Meja Lipat 200 x 80": "Meja lipat 200 x 80",
    "LEMARI": "Lemari",
    "AC": "AC",
    "Ac Portable": "AC portable",
    "Ac Split": "AC split",
    "Ac AHU": "AC AHU",
    "Storage, HDD, SSD": "Storage, HDD, SSD",
    "HDD, NAS, External HDD, Storage, SSD": "HDD, NAS, external HDD, storage, SSD",
    "UPS, Rak Server": "UPS, rak server",
    "Gedung": "Gedung",
    "Drone, Gimbal & Camera": "Drone, gimbal & camera",
    "Action Camera": "Action camera",
    "Monitor LED": "Monitor LED",
    "Monitor, Gadget": "Monitor, gadget",
    "Projector, LCD,LED Monitor": "Projector, LCD, LED monitor",
    "Tempat Kerja RTU": "Tempat kerja RTU",
    "Rakitan dan Pembuatan Manual": "Rakitan dan pembuatan manual",
    "SCAFFOLDING": "Scaffolding",
    "Pembuatan Las dll": "Pembuatan las, dll.",
    "Gengset": "Genset",
    "PLTS": "PLTS",
    "Dispenser": "Dispenser",
    "Kursi Bar": "Kursi bar",
    "shofee, tokopedia, buka lapak dll": "Shopee, Tokopedia, Bukalapak, dll.",
    "Mesin Kopi": "Mesin kopi",
    "Firewall": "Firewall",
    "SPEAKER GEDUNG": "Speaker gedung",
    "HEAT DETECTOR": "Heat detector",
    "Apar": "APAR",
    "APAR": "APAR",
    "RAM, SSD, Micro SD, FLASHDISK": "RAM, SSD, microSD, flashdisk",
    "SMOKE DETECTOR": "Smoke detector",
    "Glasboard": "Glassboard",
    "Mesin Blower": "Mesin blower",
    "Speaker": "Speaker",
    "Speaker Aktif": "Speaker aktif",
    "Spiker Aktif": "Speaker aktif",
    "Sofa Kulit": "Sofa kulit",
    "Pengarum ruangan": "Pengharum ruangan",
    "Kotak P3k": "Kotak P3K",
    "Alat2 rumah tangga": "Alat-alat rumah tangga",
    "Kursi": "Kursi",
    "storage, microsd, Flashdisk": "Storage, microSD, flashdisk",
    "Mobil Kendaraan, Pajero": "Mobil kendaraan, Pajero",
    "Mesin Penghancur Kertas": "Mesin penghancur kertas",
    "Alat Bor": "Alat bor",
    "Alat Tester": "Alat tester",
    "Senter": "Senter",
    "UNTUK NGECAT": "Untuk mengecat",
    "TV": "TV",
    "Mobil": "Mobil",
    "Helm motor": "Helm motor",
    "Jas Hujan": "Jas hujan"
}

# Descriptions to clear (junk IPs, dummy test values)
DESCRIPTION_CLEAR_IDS = {3, 3156, 78, 3149}

def clean_description(ams_id, raw_desc):
    if ams_id in DESCRIPTION_CLEAR_IDS:
        return None
    if pd.isna(raw_desc) or str(raw_desc).strip() in ["NULL", "null", ""]:
        return None
    s = str(raw_desc).strip()
    if s in DESCRIPTION_SENTENCE_CASE:
        return DESCRIPTION_SENTENCE_CASE[s]
    # Sentence case fallback: capitalize first character, rest lowercase
    return s[:1].upper() + s[1:].lower()

# ---------------------------------------------------------
# 6. Build Staging, Mapping, and Final DataFrames
# ---------------------------------------------------------
df_work = df_raw.copy()
df_work["id_int"] = df_work["id"].astype(int)

# Normalize create_date to YYYY-MM-DD HH:MM:SS
df_work["created_at"] = pd.to_datetime(df_work["create_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
df_work["updated_at"] = df_work["created_at"]

# Map of absorbed AMS IDs per primary parent: primary_ams_id -> list of all absorbed ams_ids
absorbed_ids = {}
for aid in df_work["id_int"]:
    absorbed_ids[aid] = [aid]

# Add merged typo duplicates into target
for src_id, tgt_id in MERGE_TARGET_MAP.items():
    if tgt_id in absorbed_ids and src_id not in absorbed_ids[tgt_id]:
        absorbed_ids[tgt_id].append(src_id)

# Add relegated products into parent brand
for prod_id, parent_id in RELEGATE_PRODUCT_MAP.items():
    if parent_id in absorbed_ids and prod_id not in absorbed_ids[parent_id]:
        absorbed_ids[parent_id].append(prod_id)

# Identify secondary merged rows (exact typo duplicates & consolidations)
merged_source_ids = set(MERGE_TARGET_MAP.keys())

# Build Staging records
staging_rows = []
sequential_id = 1
assigned_smart_ids = {} # ams_id -> smart_id

for idx, r in df_work.iterrows():
    aid = r["id_int"]
    if aid in merged_source_ids:
        continue # Omit secondary merged rows from master staging brand list
    
    clean_name = clean_brand_name(r["brand"])
    desc = clean_description(aid, r["description"])
    is_non_brand = aid in NON_ACTUAL_BRAND_REASONS
    ams_id_list = absorbed_ids.get(aid, [aid])
    is_relegated_parent = len(ams_id_list) > 1
    
    assigned_smart_ids[aid] = sequential_id
    
    note = ""
    if is_non_brand:
        note = NON_ACTUAL_BRAND_REASONS[aid]
    elif is_relegated_parent:
        absorbed_names = [df_work[df_work["id_int"] == x]["brand"].iloc[0] for x in ams_id_list if x != aid]
        note = f"Absorbed legacy entries: {', '.join(absorbed_names)}"
    
    staging_rows.append({
        "id": sequential_id,
        "name": clean_name,
        "description": desc,
        "ams_id": ams_id_list,
        "not_actual_brand": is_non_brand,
        "is_relegated": is_relegated_parent,
        "relegated_count": len(ams_id_list) - 1,
        "created_at": r["created_at"],
        "updated_at": r["updated_at"],
        "notes": note
    })
    sequential_id += 1

df_staging = pd.DataFrame(staging_rows)

# Build 1-to-1 Mapping Table for ALL 188 raw rows
mapping_rows = []
for idx, r in df_work.iterrows():
    aid = r["id_int"]
    raw_name = r["brand"]
    
    # Determine target smart id
    target_ams_id = aid
    is_merged = False
    is_relegated_prod = False
    
    if aid in MERGE_TARGET_MAP:
        target_ams_id = MERGE_TARGET_MAP[aid]
        is_merged = True
    elif aid in RELEGATE_PRODUCT_MAP:
        target_ams_id = RELEGATE_PRODUCT_MAP[aid]
        is_relegated_prod = True
        
    smart_id = assigned_smart_ids.get(target_ams_id, None)
    target_clean_name = clean_brand_name(df_work[df_work["id_int"] == target_ams_id]["brand"].iloc[0])
    
    mapping_type = "exact"
    if is_merged:
        mapping_type = "consolidation_merge" if aid in [3099, 3107] else "typo_merge"
    elif is_relegated_prod:
        mapping_type = "product_relegation"
    elif aid in NON_ACTUAL_BRAND_REASONS:
        mapping_type = "non_actual_brand"
        
    mapping_rows.append({
        "ams_id": aid,
        "ams_brand_name": raw_name,
        "smart_brand_id": smart_id,
        "smart_brand_name": target_clean_name,
        "is_relegated": is_relegated_prod or is_merged,
        "not_actual_brand": aid in NON_ACTUAL_BRAND_REASONS,
        "mapping_type": mapping_type,
        "notes": NON_ACTUAL_BRAND_REASONS.get(aid, "Valid brand")
    })

df_mapping = pd.DataFrame(mapping_rows)

# Build Relegated & Non-Brand Audit Table for Asset Team Review
relegated_rows = []
# 1. Product relegations
for prod_id, parent_id in RELEGATE_PRODUCT_MAP.items():
    prod_row = df_work[df_work["id_int"] == prod_id].iloc[0]
    parent_row = df_work[df_work["id_int"] == parent_id].iloc[0]
    relegated_rows.append({
        "ams_id": prod_id,
        "ams_brand_name": prod_row["brand"],
        "ams_description": prod_row["description"],
        "target_ams_id": parent_id,
        "target_brand_name": clean_brand_name(parent_row["brand"]),
        "issue_type": "product_relegation",
        "action_taken": f"Relegated under parent brand {clean_brand_name(parent_row['brand'])} (Smart ID: {assigned_smart_ids[parent_id]})",
        "discussion_recommendation": "Confirm whether this item should be tracked as a product model or separate subcategory rather than brand."
    })

# 2. Duplicate merges and consolidations
for src_id, tgt_id in MERGE_TARGET_MAP.items():
    src_row = df_work[df_work["id_int"] == src_id].iloc[0]
    tgt_row = df_work[df_work["id_int"] == tgt_id].iloc[0]
    issue_type = "consolidation" if src_id in [3099, 3107, 3159] else "duplicate_typo"
    relegated_rows.append({
        "ams_id": src_id,
        "ams_brand_name": src_row["brand"],
        "ams_description": src_row["description"],
        "target_ams_id": tgt_id,
        "target_brand_name": clean_brand_name(tgt_row["brand"]),
        "issue_type": issue_type,
        "action_taken": f"Consolidated into {clean_brand_name(tgt_row['brand'])} (Smart ID: {assigned_smart_ids[tgt_id]})",
        "discussion_recommendation": "Permanent merge into target brand."
    })

# 3. Non-actual brands (categories, dummy)
for aid, reason in NON_ACTUAL_BRAND_REASONS.items():
    if aid in RELEGATE_PRODUCT_MAP or aid in MERGE_TARGET_MAP:
        continue
    r = df_work[df_work["id_int"] == aid].iloc[0]
    relegated_rows.append({
        "ams_id": aid,
        "ams_brand_name": r["brand"],
        "ams_description": r["description"],
        "target_ams_id": None,
        "target_brand_name": None,
        "issue_type": "non_actual_brand",
        "action_taken": f"Flagged not_actual_brand = True ({reason})",
        "discussion_recommendation": "Discuss with asset team to reclassify into category, subcategory, or remove if dummy data."
    })

df_relegated = pd.DataFrame(relegated_rows)

# Build Production Final DataFrame (Strict Clean Master: not_actual_brand == False)
df_final = df_staging[df_staging["not_actual_brand"] == False][
    ["id", "name", "description", "created_at", "updated_at"]
].copy().reset_index(drop=True)

# Re-index final production IDs sequentially 1..N
df_final["id"] = range(1, len(df_final) + 1)

print(f"Staging brands prepared:        {len(df_staging)} rows")
print(f"1-to-1 Mapping rows (all AMS):  {len(df_mapping)} rows")
print(f"Relegated/Audit table rows:     {len(df_relegated)} rows")
print(f"Production Final brands:        {len(df_final)} clean verified rows")
display(df_final.head(10))


Staging brands prepared:        181 rows
1-to-1 Mapping rows (all AMS):  188 rows
Relegated/Audit table rows:     16 rows
Production Final brands:        172 clean verified rows


,id,name,description,created_at,updated_at
0,1,Compaq,None,2016-12-21 09:17:47,2016-12-21 09:17:47
1,2,InFocus,None,2016-12-21 09:17:47,2016-12-21 09:17:47
2,3,Yamaha,None,2016-12-21 09:17:47,2016-12-21 09:17:47
3,4,D-Link,None,2016-12-21 09:17:47,2016-12-21 09:17:47
4,5,Rakitan,None,2016-12-21 09:17:47,2016-12-21 09:17:47
5,6,Dell,None,2016-12-21 09:17:47,2016-12-21 09:17:47
6,7,Toshiba,None,2016-12-21 09:17:47,2016-12-21 09:17:47
7,8,Fujitsu,None,2016-12-21 09:17:47,2016-12-21 09:17:47
8,9,HP,None,2016-12-21 09:17:47,2016-12-21 09:17:47
9,10,Neo,None,2016-12-21 09:17:47,2016-12-21 09:17:47


## 5. Data Validation & Quality Checks

We run rigorous assertions to verify data integrity:
1. All brand names are strictly unique and $\le 255$ characters (`VARCHAR(255)`).
2. Descriptions satisfy the $\le 255$ character limit.
3. No invalid NULLs in required columns (`id`, `name`, `created_at`, `updated_at`).
4. Every single one of the 188 legacy AMS rows is mapped in `df_mapping`.
5. Staging `ams_id` column contains valid lists of integers.


In [5]:
# 1. Check name length constraint (VARCHAR(255))
max_name_len = df_final["name"].str.len().max()
assert max_name_len <= 255, f"Error: Maximum brand name length exceeds 255 characters ({max_name_len})"

# 2. Check description length constraint (VARCHAR(255))
max_desc_len = df_final["description"].dropna().str.len().max()
assert max_desc_len <= 255, f"Error: Maximum description length exceeds 255 characters ({max_desc_len})"

# 3. Check name uniqueness in production
duplicate_names = df_final[df_final["name"].duplicated(keep=False)]
assert duplicate_names.empty, f"Error: Duplicate brand names detected in final dataset: {duplicate_names['name'].tolist()}"

# 4. Check ID uniqueness and sequential continuity
assert df_final["id"].is_monotonic_increasing, "Error: IDs are not monotonic increasing"
assert df_final["id"].duplicated().sum() == 0, "Error: Duplicate IDs found in final dataset"
assert df_final["id"].iloc[0] == 1, "Error: ID sequence does not start at 1"

# 5. Check for unexpected NULL values in required fields
null_counts = df_final[["id", "name", "created_at", "updated_at"]].isna().sum()
assert null_counts.sum() == 0, f"Error: Found NULL values in required columns: {null_counts.to_dict()}"

# 6. Check that all 188 original AMS rows are mapped
assert len(df_mapping) == len(df_raw), f"Error: Mapping table length ({len(df_mapping)}) does not match raw AMS rows ({len(df_raw)})"

print("All 6 data validation and integrity assertions PASSED successfully!")


All 6 data validation and integrity assertions PASSED successfully!


## 6. Export Outputs

We export staging files and final production files in **two separate cells** as established:
1. **Cell 10**: Staging files (`data/smart/staging/brand/`)
   - `brands.csv`: Staging master with `ams_id` list and `not_actual_brand` flag.
   - `brand_ams_mapping.csv`: Complete 1-to-1 relational lookup mapping for all 188 legacy AMS rows.
   - `relegated_brands.csv`: Detailed audit table for discussion with the asset team.
2. **Cell 11**: Final production files (`data/smart/final/brand/`)
   - `brands.csv`: Strict 5 target columns for MS SQL Server import.
   - `brands.sql`: Executable MS SQL Server script with `IDENTITY_INSERT`.


In [6]:
# ---------------------------------------------------------
# Export Staging Outputs to data/smart/staging/brand/
# ---------------------------------------------------------
staging_brand_path = STAGING_DIR / "brands.csv"
staging_map_path = STAGING_DIR / "brand_ams_mapping.csv"
staging_relegated_path = STAGING_DIR / "relegated_brands.csv"

# 1. Staging brands with ams_id lists and not_actual_brand flags
df_staging.to_csv(staging_brand_path, index=False)

# 2. 1-to-1 Relational lookup mapping table
df_mapping.to_csv(staging_map_path, index=False)

# 3. Relegated and Non-Brand audit table for asset team discussion
df_relegated.to_csv(staging_relegated_path, index=False)

print(f"Staging files successfully exported to {STAGING_DIR.resolve()}:")
print(f" 1. {staging_brand_path.name} ({len(df_staging)} rows)")
print(f" 2. {staging_map_path.name} ({len(df_mapping)} rows)")
print(f" 3. {staging_relegated_path.name} ({len(df_relegated)} rows)")


Staging files successfully exported to /home/jovyan/data/smart/staging/brand:
 1. brands.csv (181 rows)
 2. brand_ams_mapping.csv (188 rows)
 3. relegated_brands.csv (16 rows)


In [7]:
# ---------------------------------------------------------
# Export Production Outputs to data/smart/final/brand/
# ---------------------------------------------------------
final_csv_path = FINAL_DIR / "brands.csv"
final_sql_path = FINAL_DIR / "brands.sql"

# 1. Target CSV for SQL Server Import (Strict 5 Columns)
df_final.to_csv(final_csv_path, index=False)

# 2. Generate SQL Server INSERT Script
sql_statements = [
    "-- ========================================================",
    "-- Master Brands Data Migration Script for MS SQL Server",
    f"-- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "-- Target Table: brands",
    "-- ========================================================",
    "SET IDENTITY_INSERT [brands] ON;",
    "GO",
    ""
]

for _, row in df_final.iterrows():
    name_escaped = str(row['name']).replace("'", "''")
    if pd.isna(row['description']) or row['description'] is None:
        desc_val = "NULL"
    else:
        desc_escaped = str(row['description']).replace("'", "''")
        desc_val = f"N'{desc_escaped}'"
    
    stmt = (
        f"INSERT INTO [brands] ([id], [name], [description], [created_at], [updated_at]) "
        f"VALUES ({row['id']}, N'{name_escaped}', {desc_val}, '{row['created_at']}', '{row['updated_at']}');"
    )
    sql_statements.append(stmt)

sql_statements.extend([
    "",
    "GO",
    "SET IDENTITY_INSERT [brands] OFF;",
    "GO"
])

with open(final_sql_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements))

print(f"Production files successfully exported to {FINAL_DIR.resolve()}:")
print(f" 1. {final_csv_path.name} ({len(df_final)} rows)")
print(f" 2. {final_sql_path.name} ({len(df_final)} INSERT statements)")


Production files successfully exported to /home/jovyan/data/smart/final/brand:
 1. brands.csv (172 rows)
 2. brands.sql (172 INSERT statements)
